# Pipeline

In [1]:
#import libraires
import pandas as pd
import numpy as np
import os

In [2]:
def read_csv_json(file_path: str) -> tuple :
    ''' 
    Extract file_name and file_extension from file_path
    Raise an error if file_extension different from json and csv
    Store a Dataframe in {file_name}_{file_extension}_df global variable
    Return file_name and file_extension
    '''
    file_name, file_extension = os.path.splitext(file_path)
    file_extension = file_extension[1:]
    read_functions = {'csv': pd.read_csv, 'json': pd.read_json}
    read_function = read_functions.get(file_extension)
    if read_function is None:
        raise ValueError(f"Unsupported file extension: {file_extension}")
    globals()[f"{file_name}_{file_extension}_df"] = read_function(file_path, encoding="utf-8") 
    return (file_name, file_extension)

In [3]:
#Read all csv and json files in the current directory
all_files = os.listdir(os.getcwd())
csv_json_files = [file for file in all_files if file.endswith('.csv') or file.endswith('.json')]
names_extensions = list()
for i,file in enumerate(csv_json_files):
    name, ext = read_csv_json(file) #Dataframes stored in {name}_{ext}_df variables
    names_extensions.append((name, ext))

In [4]:
#Uniform date format
new_articles_json_df['date'] = pd.to_datetime(new_articles_json_df['date'], errors='coerce').dt.strftime('%d/%m/%Y')
reviews_csv_df['date'] = pd.to_datetime(reviews_csv_df['date'], errors='coerce').dt.strftime('%d/%m/%Y')

In [5]:
#merge news_articles_csv_df and new_articles_json_df
news_articles_df = pd.concat([news_articles_csv_df,new_articles_json_df])

In [6]:
df = pd.DataFrame([], columns=['movie','media_or_journal','name','mention_date'])
i = 0
movie_names = list(movies_csv_df['title'].values)
for movie in movie_names:
    nb_reviews = reviews_csv_df.shape[0]
    for j in range(nb_reviews):
        if movie in reviews_csv_df.iloc[j,1]:
            date = reviews_csv_df.iloc[j,2]
            name = reviews_csv_df.iloc[j,3]
            df.loc[i] = [movie,'media',name,date]
            i+=1
    nb_articles = news_articles_df.shape[0]
    for j in range(nb_articles):
        if movie in news_articles_df.iloc[j,1]:
            date = news_articles_df.iloc[j,2]
            name = news_articles_df.iloc[j,3]
            df.loc[i] = [movie,'journal',name,date]
            i+=1

In [7]:
df.to_json('output.json')
df

,movie,media_or_journal,name,mention_date
0,Gladiator,journal,Libération,01/01/2004
1,Snatch,media,Télé Magazine,01/10/2002
2,Snatch,journal,Le Figaro,14/10/2000
3,Snatch,journal,Le Figaro,14/12/2000
4,Le chant du loup,media,Allo Cine,01/01/2018
5,Le chant du loup,journal,Le Monde,01/01/2020
6,Titanic,media,Movies,14/12/2003
7,Titanic,journal,Le Monde,01/01/2003
8,Intouchables,journal,Libération,02/11/2011
9,Intouchables,journal,Libération,12/01/2011


# Traitement ad-hoc : Extraire depuis le json produit par la data pipeline le nom du journal qui mentionne le plus de films différents ?

In [1]:
#import libraires
import pandas as pd

In [2]:
#import json file
output = pd.read_json('output.json')
output

,movie,media_or_journal,name,mention_date
0,Gladiator,journal,Libération,01/01/2004
1,Snatch,media,Télé Magazine,01/10/2002
2,Snatch,journal,Le Figaro,14/10/2000
3,Snatch,journal,Le Figaro,14/12/2000
4,Le chant du loup,media,Allo Cine,01/01/2018
5,Le chant du loup,journal,Le Monde,01/01/2020
6,Titanic,media,Movies,14/12/2003
7,Titanic,journal,Le Monde,01/01/2003
8,Intouchables,journal,Libération,02/11/2011
9,Intouchables,journal,Libération,12/01/2011


In [3]:
df = output[output['media_or_journal']=='journal'] # on regarde que les lignes concernant des mentions dans des journaux 
df = df.drop(columns=['media_or_journal', 'mention_date']) #on supprime les colonnes inutiles media_or_journal et mention_date
df = df.drop_duplicates() # on supprime les lignes en doubles (issues de mentions dans le même journal à deux dates différentes)
df = df.groupby(by=['name']).count() #on groupe suivant les journaux et on compte le nombre de films par joural
max_nb_mentions = df['movie'].max() #nombre maximal de mentions
journal_max_film = df[df['movie'] == max_nb_mentions].index #nom de journal(aux) ayant atteint le nombre maximal de mentions

In [4]:
nb_jour = len(journal_max_film)
if nb_jour==1:
    print(f'Le journal qui mentionne le plus de films différents est {journal_max_film[0]} avec {max_nb_mentions} films.')
else:
    print(f'Les {nb_jour} journaux qui mentionnent le plus de films différents sont {journal_max_film[0]} et {journal_max_film[1]} avec {max_nb_mentions} films chacun.')

Les 2 journaux qui mentionnent le plus de films différents sont Le Monde et Libération avec 2 films chacun.
